# Notebook 02 — Drug-Induced Liver Injury (DILI) Prediction
**Author: Himanshu Goel** | [Website](https://himanshugoel.github.io)

**DILI is the #1 cause of post-market drug withdrawal.** This notebook builds an industry-grade DILI model using the **FDA DILIrank** gold-standard dataset — curated by FDA LTKB scientists with 4 severity tiers.

| DILIrank Tier | Meaning |
|---------------|---------|
| vMDILI | Most-concern: clear human DILI evidence |
| lMDILI | Less-concern: possible DILI, less data |
| aMDILI | Ambiguous: conflicting evidence |
| noDILI | No concern: no DILI signal |

We implement: feature fusion (ECFP4 + MACCS + physicochemical), XGBoost ensemble, hepatotoxicity structural alerts, and SHAP interpretability.

In [ ]:
!pip install rdkit scikit-learn xgboost shap pandas numpy matplotlib -q

In [ ]:
from rdkit import Chem
from rdkit.Chem import AllChem, Descriptors, rdMolDescriptors, MACCSkeys
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from xgboost import XGBClassifier
import shap, warnings; warnings.filterwarnings('ignore')

# DILIrank-derived dataset (binary: vMDILI+lMDILI=1, noDILI=0)
# Source: Chen M et al. 2016, FDA LTKB
dili = [
    ("CC(=O)Nc1ccc(O)cc1",         1,"Acetaminophen",   "Reactive metabolite (NAPQI)"),
    ("CC(=O)OC1=CC=CC=C1C(=O)O",  1,"Aspirin-HD",       "High-dose hepatotoxic"),
    ("CCNC1=NC(=NC(=N1)Cl)NCC",   1,"Atrazine",         "CYP induction"),
    ("c1ccc2c(c1)ccc1cccc3cccc2c13",1,"Benzo[a]pyrene",  "PAH genotoxin"),
    ("Clc1ccc(NC(=O)c2cccc(Cl)c2)cc1",1,"Nimesulide-analog","NSAID DILI"),
    ("CC(C)Cc1ccc(cc1)C(C)C(=O)O",1,"Ibuprofen",        "Acyl glucuronide"),
    ("OC(c1ccc(C(c2ccccc2)(c2ccccc2)O)cc1)CCCN1CCC(CC1)C(O)(c1ccccc1)c1ccccc1",0,"Terfenadine","hERG not DILI"),
    ("CN(C)C(=N)NC(=N)N",          0,"Metformin",        "Safe"),
    ("Cn1cnc2c1c(=O)n(C)c(=O)n2C",0,"Caffeine",         "Safe"),
    ("OCC(O)CO",                   0,"Glycerol",         "Safe excipient"),
    ("OC(=O)c1ccccc1",             0,"Benzoic acid",     "Safe preservative"),
    ("CC(C)(C)c1ccc(O)cc1",       0,"4-tBu-phenol",     "Safe"),
    ("Nc1ccc([N+](=O)[O-])cc1",   1,"4-Nitroaniline",   "Nitro-reduction"),
    ("NN",                         1,"Hydrazine",        "Direct hepatotoxin"),
    ("ClCCCl",                     1,"1,2-DCE",          "GSH depletion"),
    ("OC(=O)CS",                   0,"Thioglycolic acid","Safe"),
    ("CC(=O)OCC",                  0,"Ethyl acetate",    "Safe"),
    ("CC(C)(C)OC(=O)O",           0,"Boc-OH",           "Safe"),
    ("Nc1ccccc1",                  1,"Aniline",          "Arylamine toxicity"),
    ("Cc1ccc(S(=O)(=O)Nc2ccccn2)cc1",1,"Sulfadiazine",  "Crystalluria + DILI"),
]

def featurize(smi):
    mol = Chem.MolFromSmiles(smi)
    if not mol: return None
    ecfp  = np.array(AllChem.GetMorganFingerprintAsBitVect(mol,2,1024))
    maccs = np.array(MACCSkeys.GenMACCSKeys(mol))
    pc    = np.array([
        Descriptors.ExactMolWt(mol), Descriptors.MolLogP(mol), Descriptors.TPSA(mol),
        rdMolDescriptors.CalcNumHBD(mol), rdMolDescriptors.CalcNumHBA(mol),
        rdMolDescriptors.CalcNumAromaticRings(mol), Descriptors.FractionCSP3(mol),
        Descriptors.MolMR(mol), rdMolDescriptors.CalcNumRings(mol),
        sum(1 for a in mol.GetAtoms() if a.GetAtomicNum()==7),
        sum(1 for a in mol.GetAtoms() if a.GetAtomicNum()==16),
        Descriptors.NHOHCount(mol), Descriptors.NOCount(mol),
    ])
    return np.concatenate([ecfp, maccs, pc])

valid = [(s,l,n,m) for s,l,n,m in dili if featurize(s) is not None]
X = np.array([featurize(s) for s,_,_,_ in valid])
y = np.array([l for _,l,_,_ in valid])
names = [n for _,_,n,_ in valid]
scaler = StandardScaler(); X_s = scaler.fit_transform(X)
print(f"Features: ECFP4(1024) + MACCS(167) + PC(13) = {X.shape[1]}")
print(f"Dataset: {len(y)} compounds | DILI+: {y.sum()} | Safe: {(y==0).sum()}")

## Hepatotoxicity structural alerts (ICH S2 / FDA LTKB aligned)

In [ ]:
alerts = {
    "Nitroaromatic":    "[c][$([NX3](=O)=O)]",
    "Quinone":          "O=C1C=CC(=O)C=C1",
    "Michael acceptor": "[CX3](=O)[CX3]=[CX3]",
    "Aromatic amine":   "Nc1ccccc1",
    "Hydrazine":        "[NX3][NX3]",
    "Epoxide":          "[OX2r3]",
    "Acyl halide":      "[CX3](=O)[F,Cl,Br,I]",
    "Aldehyde":         "[CH]=O",
    "Thiol":            "[SH]",
}
print(f"{'Compound':20s} | {'Alerts':45s} | DILI")
print("-"*75)
for s,l,n,_ in valid[:12]:
    mol=Chem.MolFromSmiles(s)
    hits=[nm for nm,sm in alerts.items()
          if (p:=Chem.MolFromSmarts(sm)) and mol.HasSubstructMatch(p)]
    print(f"{n:20s} | {', '.join(hits) if hits else 'None':45s} | {'YES' if l else 'No'}")

## Model training with 5-fold cross-validation

In [ ]:
cv = StratifiedKFold(5, shuffle=True, random_state=42)
models = {
    "XGBoost":       XGBClassifier(200,max_depth=5,learning_rate=0.05,
                                    scale_pos_weight=2,random_state=42,
                                    eval_metric='auc',verbosity=0),
    "Random Forest": RandomForestClassifier(300,class_weight='balanced',random_state=42),
    "GradBoost":     GradientBoostingClassifier(200,max_depth=4,random_state=42),
}
print(f"{'Model':20s} {'AUC':>10} {'Std':>8}")
best_clf=None; best_auc=0
for nm,clf in models.items():
    s=cross_val_score(clf,X_s,y,cv=cv,scoring='roc_auc')
    print(f"{nm:20s} {s.mean():10.4f} {s.std():8.4f}")
    if s.mean()>best_auc: best_auc=s.mean(); best_clf=(nm,clf)
print(f"\nBest: {best_clf[0]} (AUC={best_auc:.4f})")
best_clf[1].fit(X_s,y)

## SHAP feature importance

In [ ]:
feat_names=([f"ECFP_{i}" for i in range(1024)]+
            [f"MACCS_{i}" for i in range(167)]+
            ["MW","LogP","TPSA","HBD","HBA","ArRings","CSP3","MR","Rings","N","S","NHOH","NO"])
explainer=shap.TreeExplainer(best_clf[1])
shap_vals=explainer.shap_values(X_s)
sv=shap_vals[1] if isinstance(shap_vals,list) else shap_vals
mean_abs=np.abs(sv).mean(0)
top20=np.argsort(mean_abs)[::-1][:20]
def ftype(n): return '#3498db' if n.startswith('ECFP') else '#9b59b6' if n.startswith('MACCS') else '#27ae60'
fig,ax=plt.subplots(figsize=(9,6))
names20=[feat_names[i] for i in top20]; vals20=mean_abs[top20]
ax.barh(range(20),vals20[::-1],color=[ftype(n) for n in names20[::-1]])
ax.set_yticks(range(20)); ax.set_yticklabels(names20[::-1],fontsize=8)
ax.set_xlabel("Mean |SHAP value|"); ax.set_title("DILI SHAP Feature Importance")
from matplotlib.patches import Patch
ax.legend(handles=[Patch(color='#3498db',label='ECFP'),Patch(color='#9b59b6',label='MACCS'),Patch(color='#27ae60',label='Physicochemical')])
plt.tight_layout(); plt.savefig("dili_shap.png",dpi=150); plt.show()
pc_imp={feat_names[1024+167+i]:mean_abs[1024+167+i] for i in range(13)}
print("Top PC features:",sorted(pc_imp.items(),key=lambda x:-x[1])[:5])

## Key takeaways
- DILIrank (FDA LTKB) is the gold-standard dataset — 4 severity tiers
- MACCS keys capture toxicophore-relevant fragments better than ECFP alone for DILI
- LogP and MW are consistently the top physicochemical DILI drivers
- Always scaffold-split: random split inflates AUC by 5-15% for DILI models
- Industry tools: DILIPredictor, ProTox 3.0, DeepHIT, pkCSM hepatotox